### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp, desc, rank, count, when, col, sum


### Step 1 - Get the dfs

In [0]:
race_results_df = spark.read.format("parquet").load(f"{presentation_folder_path}/race_results")

### Step 2 - Transform data

In [0]:
agg_df = (
    race_results_df
    .groupBy("race_year", "driver_name", "driver_nationality", "team")
    .agg(
        sum("points").alias("total_points"),
        count(when(col("position") == 1, True)).alias("wins")
    )
)

In [0]:

driver_rank_spec = (
    Window.partitionBy("race_year")
          .orderBy(desc("total_points"), desc("wins"))
)

In [0]:

driver_standings_df = agg_df.withColumn("rank", rank().over(driver_rank_spec))

### Step 3 - Write data

In [0]:
driver_standings_df.write.format("parquet").mode("overwrite").save(f"{presentation_folder_path}/driver_standings")